# Silver Layer — tratamento dos dados

Neste notebook vamos ler as tabelas da camada Bronze e criar as tabelas da camada Silver. Nesta etapa faremos os tratamentos necessários nos dados, como a remoção de registros repetidos, a padronização dos textos, a conversão de datas e valores e o cruzamento entre as bases.

Como mais de uma pessoa do grupo pode executar o notebook, usamos `overwrite` para não acumular os mesmos registros mais de uma vez.

In [0]:
catalog = "v_credit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"

In [0]:
from datetime import datetime

from pyspark.sql import Row
from pyspark.sql import functions as F

dq_results = []


def dq_check(table_name, check_name, df, condition):
    total = df.count()
    failed = df.filter(~F.coalesce(condition, F.lit(False))).count()
    passed = failed == 0

    dq_results.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_rows=total,
            failed_rows=failed,
            passed=passed,
            checked_at=datetime.now(),
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")


def dq_check_unique(table_name, check_name, df, key_cols):
    total = df.count()
    duplicates = df.groupBy(*key_cols).count().filter(F.col("count") > 1).count()
    passed = duplicates == 0

    dq_results.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_rows=total,
            failed_rows=duplicates,
            passed=passed,
            checked_at=datetime.now(),
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {duplicates} chaves duplicadas")

## 1. Atendentes

Na base de atendentes vamos conferir se existe um registro por identificador, converter as datas e padronizar o status. O nível de atendimento aparece de formas diferentes, como `1`, `Nível 1` e `N2`, então esses valores também serão padronizados. A data de desligamento pode continuar nula quando o atendente ainda estiver ativo.

In [0]:
df_bronze_atendentes = spark.table(f"{bronze_schema}.tb_base_atendentes")

df_silver_atendentes = (
    df_bronze_atendentes
    .filter(F.col("id_atendente").isNotNull())
    .withColumn("id_atendente", F.trim(F.col("id_atendente")))
    .withColumn("nome_atendente", F.trim(F.col("nome_atendente")))
    .withColumn("nivel_atendimento", F.lower(F.trim(F.col("nivel_atendimento"))))
    .withColumn("nivel_atendimento",
        F.when(F.col("nivel_atendimento").isin("1", "nível 1"), 1)
        .when(F.col("nivel_atendimento").isin("2", "nível 2", "n2"), 2)
        .otherwise(None)
    )
    .withColumn("data_admissao", F.to_date(F.col("data_admissao"), "dd/MM/yyyy"))
    .withColumn("data_desligamento", F.to_date(F.col("data_desligamento"), "dd/MM/yyyy"))
    .withColumn("status_atendente", F.lower(F.trim(F.col("status_atendente"))))
)

dq_check_unique("silver.tb_atendentes", "id_atendente único", df_silver_atendentes, ["id_atendente"])
dq_check("silver.tb_atendentes", "nível de atendimento válido", df_silver_atendentes, F.col("nivel_atendimento").isNotNull() & F.col("nivel_atendimento").isin(1, 2))

df_silver_atendentes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_atendentes")

display(df_silver_atendentes.limit(10))

[PASS] silver.tb_atendentes | id_atendente único | 0 chaves duplicadas
[PASS] silver.tb_atendentes | nível de atendimento válido | 0/25 linhas falharam


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente
1,Marcela França,1,2019-04-18,2025-11-16,inativo
2,Zilda Esteves,1,2023-09-12,2026-01-24,inativo
3,William Marques,1,2019-05-28,2025-04-15,inativo
4,Solange Silveira,1,2019-10-07,null,ativo
5,Joaquim Barros,1,2020-12-06,2026-05-23,inativo
6,Roberta Assunção,1,2021-10-29,null,ativo
7,Luiza Galvão,1,2022-04-24,null,ativo
8,Josefa Brito,1,2023-01-25,null,ativo
9,Jaqueline Campos,1,2019-04-19,null,ativo
10,Marcos Torres,1,2020-01-20,null,ativo


## 2. Motivos, canais e clientes

Essas três tabelas serão usadas como apoio nos próximos cruzamentos. Primeiro verificamos se as chaves são únicas; não removemos registros apenas pelo identificador, pois isso poderia escolher uma linha sem uma regra definida. A data de cadastro dos clientes aparece nos formatos `dd/MM/yyyy` e `yyyy-MM-dd`, então tentamos as duas conversões. As idades vazias ou iguais a zero serão mantidas e contabilizadas somente como diagnóstico, pois o case não define uma regra para corrigi-las.

In [0]:
df_bronze_motivos = spark.table(f"{bronze_schema}.tb_base_motivos")
df_bronze_canais = spark.table(f"{bronze_schema}.tb_canais")
df_bronze_clientes = spark.table(f"{bronze_schema}.tb_clientes")

df_silver_motivos = (
    df_bronze_motivos
    .filter(F.col("id_motivo").isNotNull())
    .withColumn("id_motivo", F.trim(F.col("id_motivo")))
    .withColumn("nome_motivo", F.initcap(F.lower(F.trim(F.col("nome_motivo")))))
    .withColumn("categoria", F.initcap(F.lower(F.trim(F.col("categoria")))))
    .withColumn("criticidade", F.initcap(F.lower(F.trim(F.col("criticidade")))))
)

df_silver_canais = (
    df_bronze_canais
    .filter(F.col("id_canal").isNotNull())
    .withColumn("id_canal", F.trim(F.col("id_canal")))
    .withColumn("nome_canal", F.initcap(F.lower(F.trim(F.col("nome_canal")))))
    .withColumn("tipo_canal", F.initcap(F.lower(F.trim(F.col("tipo_canal")))))
    .withColumn("status_canal", F.lower(F.trim(F.col("status_canal"))))
)

df_silver_clientes = (
    df_bronze_clientes
    .filter(F.col("id_cliente").isNotNull())
    .withColumn("id_cliente", F.trim(F.col("id_cliente")))
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("regiao", F.initcap(F.lower(F.trim(F.col("regiao")))))
    .withColumn("idade", F.col("idade").cast("int"))
    .withColumn("data_cadastro",
        F.coalesce(
            F.try_to_date(F.col("data_cadastro"), "dd/MM/yyyy"),
            F.try_to_date(F.col("data_cadastro"), "yyyy-MM-dd"),
        )
    )
    .withColumn("dia_vencimento_fatura", F.col("dia_vencimento_fatura").cast("int"))
)

dq_check_unique("silver.tb_motivos", "id_motivo único", df_silver_motivos, ["id_motivo"])
dq_check_unique("silver.tb_canais", "id_canal único", df_silver_canais, ["id_canal"])
dq_check_unique("silver.tb_clientes", "id_cliente único", df_silver_clientes, ["id_cliente"])
dq_check("silver.tb_clientes", "data de cadastro convertida", df_silver_clientes, F.col("data_cadastro").isNotNull())
dq_check("silver.tb_clientes", "idade preenchida (diagnóstico)", df_silver_clientes, F.col("idade").isNotNull())
dq_check("silver.tb_clientes", "idade maior que zero (premissa de plausibilidade)", df_silver_clientes, F.col("idade").isNotNull() & (F.col("idade") > 0))

df_silver_motivos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_motivos")
df_silver_canais.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_canais")
df_silver_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_clientes")

display(df_silver_motivos)
display(df_silver_canais)
display(df_silver_clientes.limit(10))

[PASS] silver.tb_motivos | id_motivo único | 0 chaves duplicadas
[PASS] silver.tb_canais | id_canal único | 0 chaves duplicadas
[PASS] silver.tb_clientes | id_cliente único | 0 chaves duplicadas
[PASS] silver.tb_clientes | data de cadastro convertida | 0/47705 linhas falharam
[FAIL] silver.tb_clientes | idade preenchida (diagnóstico) | 459/47705 linhas falharam
[FAIL] silver.tb_clientes | idade maior que zero (premissa de plausibilidade) | 706/47705 linhas falharam


id_motivo,nome_motivo,categoria,criticidade
11,Bloqueio De Cartão,Cartao,Alta
7,Compra Não Autorizada,Cartao,Alta
13,Dúvidas Gerais Sobre Programa De Pontos,Beneficios,Baixa
14,Transferência De Agência,null,Media
9,Problema Com Aplicativo,Atendimento,Media
8,Consulta De Contrato,Financeiro,Media
15,Renegociação De Dívida,Financeiro,Alta
16,Cancelamento De Conta,null,Alta
4,"Alteração De Dados Cadastrais (vencimento Da Fatura, Telefone, Email)",Cadastral,Baixa
3,Contestação De Fatura,Financeiro,Alta


id_canal,nome_canal,tipo_canal,status_canal
1,Ura,Autosservico,ativo
2,Chatbot,Autosservico,ativo
3,Atendimento Inicial,Humano,ativo
4,Atendimento Especializado,Humano,ativo


id_cliente,nome,email,regiao,idade,data_cadastro,dia_vencimento_fatura
56324416706,Sérgio Martins Miranda,sergio.miranda98@hotmail.com,Sul,51,2024-10-14,10
97653229900,Leandro Magalhães Lopes,lopes.leandro@yahoo.com.br,Sudeste,51,2026-07-07,10
15132904850,Giovana Soares Vasconcelos,giovana.vasconcelos26@hotmail.com,Sudeste,52,2019-02-16,25
62566924227,Adriana Nascimento,adriana96@gmail.com,Nordeste,39,2025-08-27,20
35213062014,Vera Duarte,vera.duarte@ig.com.br,Sudeste,34,2026-05-07,15
11173485227,Renata Freitas Fernandes,renata96@yahoo.com.br,Sudeste,37,2025-01-26,5
50855203771,Lucas Cardoso,lucascardoso@uol.com.br,Nordeste,38,2026-07-15,28
72534560781,Camila Guimarães Gomes,camila_gomes@uol.com.br,Nordeste,80,2025-09-03,25
40409080977,Thiago Peixoto,thiago98@gmail.com,Sul,92,2023-06-21,25
65936173686,Mariana Aparecida Esteves,esteves.mariana@ig.com.br,Norte,41,2025-05-31,15


## 3. Horários dos chamados

Em `chamados.csv`, os campos de abertura e finalização estão vazios e o campo de início contém apenas textos como `igual a hora de abertura`. Por isso, usamos `Chamados_Hora.CSV` para obter os horários reais. Quando o início estiver vazio, o chamado será mantido com os tempos calculados nulos.

In [0]:
df_bronze_chamados_hora = spark.table(f"{bronze_schema}.tb_chamados_hora")

df_silver_chamados_hora = (
    df_bronze_chamados_hora
    .filter(F.col("ID_Chamado").isNotNull())
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente_horario")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento")
    .withColumn("hora_abertura_chamado", F.to_timestamp(F.col("hora_abertura_chamado"), "dd/MM/yyyy 'às' HH:mm:ss"))
    .withColumn("hora_inicio_atendimento", F.to_timestamp(F.col("hora_inicio_atendimento"), "dd/MM/yyyy 'às' HH:mm:ss"))
    .withColumn("hora_finalizacao_atendimento", F.to_timestamp(F.col("hora_finalizacao_atendimento"), "dd/MM/yyyy 'às' HH:mm:ss"))
    .withColumn("tempo_espera_calculado",
        F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn("tempo_atendimento_calculado",
        F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long"))
)

dq_check_unique("silver.tb_chamados_hora", "id_chamado único", df_silver_chamados_hora, ["id_chamado"])
dq_check("silver.tb_chamados_hora", "horário de abertura convertido", df_silver_chamados_hora, F.col("hora_abertura_chamado").isNotNull())
dq_check("silver.tb_chamados_hora", "horário de início preenchido", df_silver_chamados_hora, F.col("hora_inicio_atendimento").isNotNull())
dq_check("silver.tb_chamados_hora", "horário de finalização convertido", df_silver_chamados_hora, F.col("hora_finalizacao_atendimento").isNotNull())
dq_check("silver.tb_chamados_hora", "tempo de espera não negativo", df_silver_chamados_hora, F.coalesce(F.col("tempo_espera_calculado") >= 0, F.lit(True)))
dq_check("silver.tb_chamados_hora", "tempo de atendimento não negativo", df_silver_chamados_hora, F.coalesce(F.col("tempo_atendimento_calculado") >= 0, F.lit(True)))

df_silver_chamados_hora.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_chamados_hora")

display(df_silver_chamados_hora.limit(10))

[PASS] silver.tb_chamados_hora | id_chamado único | 0 chaves duplicadas
[PASS] silver.tb_chamados_hora | horário de abertura convertido | 0/131838 linhas falharam
[FAIL] silver.tb_chamados_hora | horário de início preenchido | 3414/131838 linhas falharam
[PASS] silver.tb_chamados_hora | horário de finalização convertido | 0/131838 linhas falharam
[FAIL] silver.tb_chamados_hora | tempo de espera não negativo | 192/131838 linhas falharam
[FAIL] silver.tb_chamados_hora | tempo de atendimento não negativo | 1/131838 linhas falharam


id_chamado,id_cliente_horario,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,tempo_espera_calculado,tempo_atendimento_calculado
121151,25203409261,2026-06-29T14:12:48.000Z,2026-06-29T14:21:24.000Z,2026-06-29T15:06:16.000Z,516,2692
48388,99076736475,2025-09-16T09:03:42.000Z,2025-09-16T09:07:41.000Z,2025-09-16T10:11:50.000Z,239,3849
12605,13153799751,2025-03-03T14:02:31.000Z,2025-03-03T14:02:31.000Z,2025-03-03T14:04:25.000Z,0,114
120663,13276442322,2026-06-08T15:38:20.000Z,2026-06-08T15:38:20.000Z,2026-06-08T15:40:18.000Z,0,118
45377,74032831005,2025-09-23T14:41:45.000Z,2025-09-23T14:41:45.000Z,2025-09-23T14:43:35.000Z,0,110
96072,12791762924,2026-04-02T05:52:42.000Z,2026-04-02T05:52:42.000Z,2026-04-02T05:55:01.000Z,0,139
90851,18601632897,2026-03-17T09:04:43.000Z,2026-03-17T09:04:43.000Z,2026-03-17T09:06:41.000Z,0,118
100413,68824391895,2026-04-18T13:24:43.000Z,2026-04-18T13:26:20.000Z,2026-04-18T13:45:47.000Z,97,1167
41450,65736093750,2025-08-03T08:44:59.000Z,2025-08-03T09:14:42.000Z,2025-08-03T10:12:10.000Z,1783,3448
122039,53817300416,2026-06-24T19:47:08.000Z,2026-06-24T19:47:08.000Z,2026-06-24T19:51:18.000Z,0,250


## 4. Chamados

Na base principal encontramos algumas linhas completamente repetidas, por isso usamos `dropDuplicates()`. Também padronizamos os valores de resolução, canal e motivo antes de fazer os joins com as tabelas de apoio. Os tempos informados e calculados são mantidos em colunas separadas para que suas diferenças apareçam nas verificações. Também conferimos se cliente e atendente existem nas respectivas bases e se o cliente é o mesmo nos dois arquivos de chamados.

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")

df_silver_chamados = (
    df_bronze_chamados
    .dropDuplicates()
    .filter(F.col("id_chamado").isNotNull())
    .withColumn("motivo", F.lower(F.trim(F.col("motivo"))))
    .withColumn("canal", F.lower(F.trim(F.col("canal"))))
    .withColumn("canal", F.regexp_replace(F.col("canal"), r"[._]+", " "))
    .withColumn("canal", F.regexp_replace(F.col("canal"), r"\s+", " "))
    .withColumn("canal",
        F.when(F.col("canal").isin("u r a", "ura"), "ura")
        .when(F.col("canal").isin("chat bot", "chatbot"), "chatbot")
        .when(F.col("canal").isin("atend inicial", "atendimento inicial"), "atendimento inicial")
        .when(F.col("canal").isin("atend especializado", "atendimento especializado"), "atendimento especializado")
        .otherwise(F.col("canal"))
    )
    .withColumn("resolvido", F.lower(F.trim(F.col("resolvido"))))
    .withColumn("resolvido",
        F.when(F.col("resolvido").isin("sim", "s", "1"), "Sim")
        .when(F.col("resolvido").isin("não", "nao", "n", "0", "nao resolvido"), "Não")
        .otherwise("Não informado")
    )
    .withColumn("tempo_espera", F.when(F.upper(F.col("tempo_espera")) == "NULL", None).otherwise(F.col("tempo_espera")).cast("long"))
    .withColumn("tempo_atendimento", F.when(F.upper(F.col("tempo_atendimento")) == "NULL", None).otherwise(F.col("tempo_atendimento")).cast("long"))
    .withColumnRenamed("tempo_espera", "tempo_espera_informado")
    .withColumnRenamed("tempo_atendimento", "tempo_atendimento_informado")
    .drop("hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento")
)

df_motivos_join = df_silver_motivos.withColumn("motivo", F.lower(F.trim(F.col("nome_motivo"))))
df_canais_join = df_silver_canais.withColumn("canal", F.lower(F.trim(F.col("nome_canal"))))

df_clientes_referencia = df_silver_clientes.select("id_cliente").withColumn("cliente_cadastrado", F.lit(True))
df_atendentes_referencia = df_silver_atendentes.select("id_atendente").withColumn("atendente_cadastrado", F.lit(True))

df_silver_chamados = (
    df_silver_chamados
    .join(df_motivos_join.select("id_motivo", "motivo"), on="motivo", how="left")
    .join(df_canais_join.select("id_canal", "canal"), on="canal", how="left")
    .join(df_silver_chamados_hora, on="id_chamado", how="left")
    .join(df_clientes_referencia, on="id_cliente", how="left")
    .join(df_atendentes_referencia, on="id_atendente", how="left")
    .withColumn("cliente_cadastrado", F.coalesce(F.col("cliente_cadastrado"), F.lit(False)))
    .withColumn("atendente_cadastrado", F.coalesce(F.col("atendente_cadastrado"), F.lit(False)))
    .withColumn("cliente_horario_consistente", F.col("id_cliente") == F.col("id_cliente_horario"))
    .withColumn("tempo_espera_consistente", F.col("tempo_espera_informado") == F.col("tempo_espera_calculado"))
    .withColumn("tempo_atendimento_consistente", F.col("tempo_atendimento_informado") == F.col("tempo_atendimento_calculado"))
)

dq_check_unique("silver.tb_chamados", "id_chamado único", df_silver_chamados, ["id_chamado"])
dq_check("silver.tb_chamados", "motivo identificado", df_silver_chamados, F.col("id_motivo").isNotNull())
dq_check("silver.tb_chamados", "canal identificado", df_silver_chamados, F.col("id_canal").isNotNull())
dq_check("silver.tb_chamados", "cliente encontrado na base de clientes", df_silver_chamados, F.col("cliente_cadastrado"))
dq_check("silver.tb_chamados", "atendente encontrado na base de atendentes", df_silver_chamados, F.col("atendente_cadastrado"))
dq_check("silver.tb_chamados", "cliente igual nas duas bases de chamados", df_silver_chamados, F.coalesce(F.col("cliente_horario_consistente"), F.lit(False)))
dq_check("silver.tb_chamados", "tempo de espera consistente quando ambos estão disponíveis", df_silver_chamados, F.coalesce(F.col("tempo_espera_consistente"), F.lit(True)))
dq_check("silver.tb_chamados", "tempo de atendimento consistente quando ambos estão disponíveis", df_silver_chamados, F.coalesce(F.col("tempo_atendimento_consistente"), F.lit(True)))

df_silver_chamados.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_chamados")

display(df_silver_chamados.limit(10))

[PASS] silver.tb_chamados | id_chamado único | 0 chaves duplicadas
[FAIL] silver.tb_chamados | motivo identificado | 2236/131838 linhas falharam
[PASS] silver.tb_chamados | canal identificado | 0/131838 linhas falharam
[FAIL] silver.tb_chamados | cliente encontrado na base de clientes | 1330/131838 linhas falharam
[FAIL] silver.tb_chamados | atendente encontrado na base de atendentes | 764/131838 linhas falharam
[FAIL] silver.tb_chamados | cliente igual nas duas bases de chamados | 1330/131838 linhas falharam
[FAIL] silver.tb_chamados | tempo de espera consistente quando ambos estão disponíveis | 6180/131838 linhas falharam
[FAIL] silver.tb_chamados | tempo de atendimento consistente quando ambos estão disponíveis | 27987/131838 linhas falharam


id_atendente,id_cliente,id_chamado,canal,motivo,resolvido,tempo_espera_informado,tempo_atendimento_informado,id_motivo,id_canal,id_cliente_horario,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,tempo_espera_calculado,tempo_atendimento_calculado,cliente_cadastrado,atendente_cadastrado,cliente_horario_consistente,tempo_espera_consistente,tempo_atendimento_consistente
9,36276971799,52060,ura,renegociação de dívida,Não,null,216,15,1,36276971799,2025-10-15T22:40:51.000Z,2025-10-15T22:40:51.000Z,2025-10-15T22:44:27.000Z,0,216,true,true,true,null,true
3,52402176464,89333,chatbot,renegociação de dívida,Não,null,104,15,2,52402176464,2026-03-29T23:01:27.000Z,2026-03-29T23:01:27.000Z,2026-03-29T23:03:11.000Z,0,104,true,true,true,null,true
18,95609443006,128467,ura,problema com aplicativo,Não,null,71,9,1,95609443006,2026-07-03T17:18:53.000Z,2026-07-03T17:18:53.000Z,2026-07-03T17:20:04.000Z,0,71,true,true,true,null,true
20,21748174497,45000,chatbot,renegociação de dívida,Não informado,null,106,15,2,21748174497,2025-09-30T16:59:45.000Z,2025-09-30T16:59:45.000Z,2025-09-30T17:01:31.000Z,0,106,true,true,true,null,true
1,70649266905,97042,ura,cancelamento de conta,Não,null,218,16,1,70649266905,2026-04-24T07:27:17.000Z,2026-04-24T07:27:17.000Z,2026-04-24T07:30:55.000Z,0,218,true,true,true,null,true
13,98181934335,118384,atendimento inicial,contestação de fatura,Não,1,1,3,3,98181934335,2026-06-24T20:23:37.000Z,2026-06-24T20:42:52.000Z,2026-06-24T21:13:17.000Z,1155,1825,true,true,true,false,false
14,50297589152,44717,ura,consulta de fatura,Sim,null,271,10,1,50297589152,2025-09-18T16:37:38.000Z,2025-09-18T16:37:38.000Z,2025-09-18T16:42:09.000Z,0,271,true,true,true,null,true
11,58605012202,91912,chatbot,"alteração de dados cadastrais (vencimento da fatura, telefone, email)",Sim,null,50,4,2,58605012202,2026-03-27T15:37:17.000Z,2026-03-27T15:37:17.000Z,2026-03-27T15:38:07.000Z,0,50,true,true,true,null,true
19,39808757879,84270,atendimento especializado,problema com aplicativo,Sim,564,3,9,4,39808757879,2026-02-09T09:02:21.000Z,2026-02-09T09:11:45.000Z,2026-02-09T10:12:13.000Z,564,3628,true,true,true,true,false
19,45998304098,128920,chatbot,bloqueio de cartao,Sim,null,52,null,2,45998304098,2026-07-31T08:45:59.000Z,2026-07-31T08:45:59.000Z,2026-07-31T08:46:51.000Z,0,52,true,true,true,null,true


## 5. Custos e pesquisa de satisfação

Nos custos removemos apenas as linhas completamente repetidas e retiramos os textos `R$` e `reais` antes da conversão para decimal. Valores negativos serão mantidos, pois o case não informa se eles representam erro, estorno ou ajuste. Nas pesquisas mantemos as respostas recebidas e criamos uma coluna para indicar se a nota está entre 1 e 5, conforme a descrição da base no enunciado.

In [0]:
df_bronze_custos = spark.table(f"{bronze_schema}.tb_custos")
df_bronze_pesquisas = spark.table(f"{bronze_schema}.tb_pesquisa_satisfacao")

df_silver_custos = (
    df_bronze_custos
    .dropDuplicates()
    .filter(
        F.col("id_custo").isNotNull()
        & F.col("id_chamado").isNotNull()
    )
    .withColumn(
        "custo",
        F.lower(F.trim(F.col("custo")))
    )
    .withColumn(
        "custo",
        F.replace(
            F.col("custo"),
            F.lit("r$"),
            F.lit("")
        )
    )
    .withColumn(
        "custo",
        F.replace(
            F.col("custo"),
            F.lit("reais"),
            F.lit("")
        )
    )
    .withColumn(
        "custo",
        F.replace(
            F.col("custo"),
            F.lit(","),
            F.lit(".")
        )
    )
    .withColumn(
        "custo",
        F.trim(F.col("custo"))
    )
    .withColumn(
        "custo",
        F.col("custo").cast("decimal(10,2)")
    )
)

df_silver_pesquisas = (
    df_bronze_pesquisas
    .filter(F.col("id_pesquisa").isNotNull() & F.col("id_chamado").isNotNull())
    .withColumn("nota_atendimento",
        F.when(F.trim(F.col("nota_atendimento")) == "", None)
        .when(F.upper(F.trim(F.col("nota_atendimento"))) == "NULL", None)
        .otherwise(F.col("nota_atendimento"))
        .cast("int")
    )
    .withColumn("nota_valida", F.col("nota_atendimento").between(1, 5))
)

dq_check_unique("silver.tb_custos", "id_custo único", df_silver_custos, ["id_custo"])
dq_check("silver.tb_custos", "custo convertido para decimal", df_silver_custos, F.col("custo").isNotNull())
dq_check_unique("silver.tb_pesquisa_satisfacao", "id_pesquisa único", df_silver_pesquisas, ["id_pesquisa"])
dq_check("silver.tb_pesquisa_satisfacao", "nota entre 1 e 5", df_silver_pesquisas, F.coalesce(F.col("nota_valida"), F.lit(False)))

df_silver_custos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_custos")
df_silver_pesquisas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_pesquisa_satisfacao")

display(df_silver_custos.limit(10))
display(df_silver_pesquisas.limit(10))

[PASS] silver.tb_custos | id_custo único | 0 chaves duplicadas
[PASS] silver.tb_custos | custo convertido para decimal | 0/130288 linhas falharam
[PASS] silver.tb_pesquisa_satisfacao | id_pesquisa único | 0 chaves duplicadas
[FAIL] silver.tb_pesquisa_satisfacao | nota entre 1 e 5 | 6899/81021 linhas falharam


id_custo,id_chamado,custo
114049,110377,0.39
87679,83693,1.46
92429,88501,12.24
58033,53673,24.43
31491,26832,11.14
115079,111418,7.95
132390,128920,0.46
39350,34788,0.52
52156,47733,0.51
53892,49491,16.70


id_pesquisa,id_chamado,nota_atendimento,nota_valida
71619,115746,3,true
13534,19389,4,true
53852,83856,4,true
5563,7771,5,true
15959,22934,5,true
77687,127019,3,true
74949,121886,5,true
29267,43292,4,true
27051,39785,3,true
78680,128918,2,true


## 6. Resultado das verificações

As verificações realizadas durante o notebook são reunidas em uma tabela para facilitar a revisão do grupo.

In [0]:
df_dq_log = spark.createDataFrame(dq_results)

df_dq_log.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.dq_log")

display(df_dq_log)

table_name,check_name,total_rows,failed_rows,passed,checked_at
silver.tb_atendentes,id_atendente único,25,0,true,2026-10-28T04:07:27.721Z
silver.tb_atendentes,nível de atendimento válido,25,0,true,2026-10-28T04:07:28.669Z
silver.tb_motivos,id_motivo único,14,0,true,2026-10-28T04:08:09.965Z
silver.tb_canais,id_canal único,4,0,true,2026-10-28T04:08:10.908Z
silver.tb_clientes,id_cliente único,47705,0,true,2026-10-28T04:08:11.900Z
silver.tb_clientes,data de cadastro convertida,47705,0,true,2026-10-28T04:08:12.831Z
silver.tb_clientes,idade preenchida (diagnóstico),47705,459,false,2026-10-28T04:08:13.782Z
silver.tb_clientes,idade maior que zero (premissa de plausibilidade),47705,706,false,2026-10-28T04:08:14.654Z
silver.tb_chamados_hora,id_chamado único,131838,0,true,2026-10-28T04:09:31.086Z
silver.tb_chamados_hora,horário de abertura convertido,131838,0,true,2026-10-28T04:09:32.164Z
